# Isolation Forest

In [1]:
import sys, time, warnings
from pathlib import Path
def find_root(marker='Data/train.csv'):
    for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (d / marker).exists():
            return d
    raise FileNotFoundError(f'Không thấy repo root từ {Path.cwd()}')
ROOT = find_root()
sys.path.append(str(ROOT / 'Modeling' / 'Code'))
sys.path.append(str(ROOT / 'FE_FS' / 'Code'))
warnings.filterwarnings("ignore")

In [2]:
import numpy as np, pandas as pd
import joblib
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score
from eval_protocol import time_split_per_kpi, evaluate_protocol
from preprocess import preprocess_all
from features import build_features, select_features

In [3]:
df = pd.read_csv(ROOT / 'Data' / 'train.csv')
df.columns = ["timestamp", "value", "label", "kpi"]
SHINGLE_GRID = (8, 16, 32, 48, 64); MS_GRID = (128, 256, 512)

In [4]:
def run_kpi(gk):
    step = int(pd.Series(np.diff(np.sort(gk.timestamp.values))).mode().iloc[0])
    gk = time_split_per_kpi(gk, train_frac=0.6, val_frac=0.2)
    p = preprocess_all(gk, max_gap_points=5, norm_method="robust").sort_values("timestamp").reset_index(drop=True)
    y = p.label.values.astype(int); spl = p.split.values

    if y[spl == "val"].sum() == 0 or y[spl == "test"].sum() == 0:
        return None
    out = dict(kpi=gk.kpi.iloc[0][:8], n_anom_test=int(y[spl == "test"].sum()))

    # ---- ROUTE A: shingle ----
    v = p.value_norm.values.astype(float)
    def shingle_score(S, MS, seed):
        W = sliding_window_view(v, S) 
        cur = np.arange(S-1, len(v)) 
        ok = ~np.isnan(W).any(1)
        Xok, spok = W[ok], spl[cur][ok]
        sc = -IsolationForest(n_estimators=100, max_samples=MS, contamination="auto",
                              random_state=seed).fit(Xok[spok == "train"]).decision_function(Xok)
        
        full = np.full(len(v), np.nan) 
        full[cur[ok]] = sc
        return full
    
    bestA = None                                                 # tune trên VAL
    for S in SHINGLE_GRID:
        for MS in MS_GRID:
            s = shingle_score(S, MS, 42)
            vm = (spl == "val") & ~np.isnan(s)
            ap = average_precision_score(y[vm], s[vm]) if y[vm].sum() else -1
            if bestA is None or ap > bestA[0]: bestA = (ap, S, MS)
    _, S, MS = bestA
    s = shingle_score(S, MS, 0)
    te = (spl == "test"); validA = ~np.isnan(s)
    vm = (spl == "val") & validA; tm = te & validA
    a_AP_val = average_precision_score(y[vm], s[vm]) if y[vm].sum() else -1.0
    rA = evaluate_protocol(y[vm], s[vm], y[tm], s[tm], step_s=step,
                           y_test_full=y[te], valid_test=validA[te])

    rA = evaluate_protocol(y[vm], s[vm], y[tm], s[tm], step_s=step,
                           y_test_full=y[te], valid_test=validA[te])

    out.update(a_shingle=S, a_max_samples=MS, a_AP=round(rA["AP_pw"], 3),
               a_AP_val=round(a_AP_val, 3), a_ROC=round(rA["ROC"], 3),
               a_PW_P=round(rA["PW"]["precision"], 3), a_PW_R=round(rA["PW"]["recall"], 3),
               a_PW_F1=round(rA["PW"]["fbeta"], 3), a_thr_pw=round(rA["PW"]["threshold"], 4))

    # ---- ROUTE B: feature ----
    F = build_features(p); final = select_features(F, p)
    X = F[final].values; valid = ~np.isnan(X).any(1); tr = spl == "train"

    def feat_score(MS, seed):
        return -IsolationForest(n_estimators=100, max_samples=MS, contamination="auto",
                                random_state=seed).fit(X[tr & valid]).decision_function(X)

    bestB = None
    for MS in MS_GRID:
        s = feat_score(MS, 42); vm = (spl == "val") & valid
        ap = average_precision_score(y[vm], s[vm]) if y[vm].sum() else -1
        if bestB is None or ap > bestB[0]: bestB = (ap, MS)
    _, MSb = bestB
    clfB = IsolationForest(n_estimators=100, max_samples=MSb, contamination="auto",
                           random_state=0).fit(X[tr & valid])   # GIỮ model cho SHAP
    sc = -clfB.decision_function(X)
    te = (spl == "test")
    vm = (spl == "val") & valid; tm = te & valid
    rB = evaluate_protocol(y[vm], sc[vm], y[tm], sc[tm], step_s=step,
                           y_test_full=y[te], valid_test=valid[te])
    b_AP_val = average_precision_score(y[vm], sc[vm]) if y[vm].sum() else -1.0
    out.update(b_n_feat=len(final), b_max_samples=MSb, b_AP=round(rB["AP_pw"], 3),
               b_AP_val=round(b_AP_val, 3), b_ROC=round(rB["ROC"], 3),
               b_PW_P=round(rB["PW"]["precision"], 3), b_PW_R=round(rB["PW"]["recall"], 3),
               b_PW_F1=round(rB["PW"]["fbeta"], 3), b_thr_pw=round(rB["PW"]["threshold"], 4))
    # artifact: score route B (Panel 2) + model & feature test (SHAP, lấy mẫu ≤2000 hàng)
    Xt = X[tm]; rng = np.random.default_rng(0)
    samp = rng.choice(len(Xt), min(2000, len(Xt)), replace=False) if len(Xt) else np.array([], int)
    out["_score"] = sc[tm]; out["_ts"] = p.timestamp.values[tm]; out["_y"] = y[tm]
    out["_model"] = clfB; out["_Xtest"] = Xt[samp]; out["_features"] = final
    return out

In [5]:
ART = ROOT / 'Modeling' / 'Artifacts'; ART.mkdir(parents=True, exist_ok=True)
rows, skipped, SCORES = [], [], []
t0 = time.time()
for kpi, gk in df.groupby("kpi"):
    r = run_kpi(gk.copy())
    if r is None:
        skipped.append(kpi[:8])
    else:
        SCORES.append(pd.DataFrame({"kpi": r["kpi"], "timestamp": r.pop("_ts"),
                                    "y": r.pop("_y"), "score": r.pop("_score")}))
        joblib.dump({"model": r.pop("_model"), "X_test": r.pop("_Xtest"),
                     "features": r.pop("_features")}, ART / f"iffeat_{r['kpi']}.joblib")
        rows.append(r); print(f"done {r['kpi']} | a_AP {r['a_AP']} b_AP {r['b_AP']}")
print(f"Skipped ({len(skipped)}):", skipped)
print(f"(time {time.time()-t0:.0f}s)")

done 02e99bd4 | a_AP 0.796 b_AP 0.898
done 07927a9a | a_AP 0.038 b_AP 0.038
done 09513ae3 | a_AP 0.009 b_AP 0.013
done 18fbb1d5 | a_AP 0.116 b_AP 0.65
done 1c35dbf5 | a_AP 0.976 b_AP 0.977
done 40e25005 | a_AP 0.02 b_AP 0.217
done 71595dd7 | a_AP 0.01 b_AP 0.202
done 7c189dd3 | a_AP 0.068 b_AP 0.622
done 88cf3a77 | a_AP 0.017 b_AP 0.231
done 8bef9af9 | a_AP 0.01 b_AP 0.543
done 8c892e55 | a_AP 0.861 b_AP 0.395
done 9ee58794 | a_AP 0.072 b_AP 0.906
done a40b1df8 | a_AP 0.058 b_AP 0.677
done affb01ca | a_AP 0.006 b_AP 0.462
done c58bfcba | a_AP 0.003 b_AP 0.004
done cff6d3c0 | a_AP 0.024 b_AP 0.181
done da403e4e | a_AP 0.582 b_AP 0.767
done e0770391 | a_AP 0.874 b_AP 0.466
Skipped (8): ['046ec29d', '54e8a140', '769894ba', '76f4550c', '8a20c229', '9bd90500', 'a5bf5d65', 'b3b2e6d1']
(time 552s)


In [6]:
res = pd.DataFrame(rows).sort_values("b_AP", ascending=False)
print(f"MACRO AP    | route a = {res['a_AP'].mean():.3f} | route b = {res['b_AP'].mean():.3f}")
print(f"MACRO PW_F1 | route a = {res['a_PW_F1'].mean():.3f} | route b = {res['b_PW_F1'].mean():.3f}")
win_b = (res['b_AP'] > res['a_AP']).sum()
print(f"Route (theo AP): route b {win_b}/{len(res)} | route a {len(res)-win_b}/{len(res)}")
res

MACRO AP    | route a = 0.252 | route b = 0.458
MACRO PW_F1 | route a = 0.155 | route b = 0.352
Route (theo AP): route b 15/18 | route a 3/18


,kpi,n_anom_test,a_shingle,a_max_samples,a_AP,a_AP_val,a_ROC,a_PW_P,a_PW_R,a_PW_F1,a_thr_pw,b_n_feat,b_max_samples,b_AP,b_AP_val,b_ROC,b_PW_P,b_PW_R,b_PW_F1,b_thr_pw
4,1c35dbf5,2709,64,512,0.976,0.975,0.994,0.527,0.987,0.687,0.0426,25,512,0.977,0.978,0.998,0.590,0.998,0.742,0.0521
11,9ee58794,761,8,256,0.072,0.028,0.617,0.064,0.987,0.120,-0.0091,14,512,0.906,0.167,0.986,0.567,0.917,0.701,0.1117
0,02e99bd4,1382,48,512,0.796,0.767,0.967,0.576,0.811,0.674,-0.0003,19,256,0.898,0.742,0.981,0.654,0.889,0.754,0.0327
16,da403e4e,238,8,512,0.582,0.961,0.959,0.496,0.819,0.618,0.0552,21,512,0.767,0.853,0.993,0.219,0.979,0.358,-0.0135
12,a40b1df8,86,8,512,0.058,0.014,0.910,0.054,0.605,0.099,0.0413,15,256,0.677,0.630,0.912,0.800,0.558,0.658,0.1889
3,18fbb1d5,47,32,512,0.116,0.998,0.834,0.214,0.255,0.233,0.0036,23,512,0.650,0.992,0.906,0.143,0.681,0.236,0.0104
7,7c189dd3,63,8,512,0.068,0.023,0.932,0.083,0.524,0.143,0.0890,16,256,0.622,0.631,0.928,0.692,0.571,0.626,0.1924
9,8bef9af9,73,8,512,0.010,0.005,0.850,0.009,0.753,0.019,0.1021,15,512,0.543,0.674,0.932,0.571,0.438,0.496,0.1993
17,e0770391,2706,16,256,0.874,0.281,0.947,0.000,0.000,0.000,0.2341,15,512,0.466,0.675,0.951,0.299,0.040,0.070,0.0929
13,affb01ca,76,8,512,0.006,0.008,0.784,0.008,0.750,0.015,0.1200,15,512,0.462,0.603,0.946,0.667,0.421,0.516,0.2203


In [7]:
res.to_json(ROOT / 'Modeling' / 'ML' / 'IsolationForest' / 'if_per_kpi_config.json',
            orient="records", indent=1)
pd.concat(SCORES, ignore_index=True).to_parquet(ART / 'scores_IF_feat.parquet', index=False)
print("Đã lưu config + scores_IF_feat.parquet +", len(SCORES), "model joblib ->", ART)

Đã lưu config + scores_IF_feat.parquet + 18 model joblib -> C:\Projects\anomaly-detection-fundamentals\Modeling\Artifacts
